# Testing v2 algorithm on unseen test datasets

This notebook runs the same end-to-end deduplication workflow on the **Depression** and **Diabetes**  datasets.

The workflow is intentionally narrow:

1. Load the dataset.
2. Build blocked candidate pairs.
3. Score the pairs with the weighted deduper.
4. Evaluate at the pair level and record level.


## Evaluation strategy

We use two complementary evaluation levels:

1. **Pair-level** — classic precision/recall/F1 on the set of blocked candidate pairs; computed for a range of score thresholds.
2. **Record-level (ASySD-style)** — following Hair et al. (2023), pairs are clustered into connected components and each cluster retains exactly one record. The resulting kept/removed decisions are compared against gold-standard duplicate groups to build a record-level confusion matrix.

The record-level view is more directly interpretable: a false positive means a real unique paper was accidentally discarded, and a false negative means a real duplicate survived deduplication.

## Setup

Imports cover three areas:

- **Standard data science stack** (`pandas`, `numpy`, `sklearn`) for data handling and computing evaluation metrics.
- **App modules** — `Deduper` is the main deduplication class; `BLOCK_RULES` defines which field combinations are used for candidate-pair blocking; `GoldStandardPaper` is the Pydantic model that adds gold-standard fields (`recordid`, `duplicateid`) on top of the base `Paper` schema.
- **Path configuration** — the notebook resolves the repository root dynamically so it can be run from either the `notebooks/` folder or the repo root.

In [1]:
import sys
from collections.abc import Sequence
from pathlib import Path

import pandas as pd
from loguru import logger
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from tqdm.auto import tqdm

repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from algorithm.evaluation import SCORE_CONFIG, run_dataset_pipeline

logger.remove()
logger.add(sys.stderr, level="WARNING")

1

In [2]:
DATASETS = {
    "depression": repo_root / "notebooks" / "data" / "depression_data.csv",
    "diabetes": repo_root / "notebooks" / "data" / "diabetes_data.csv"
}

dataset_runs = {
    name: run_dataset_pipeline(name, path)
    for name, path in DATASETS.items()
}

summary_df = pd.DataFrame(
    {
        "dataset": dataset_name,
        "n_records": len(result["papers"]),
        "n_pairs": len(result["pairs_df"]),
        "pair_best_threshold": (
            None if result["best_pair"] is None else result["best_pair"]["threshold"]
        ),
        "pair_best_sensitivity": (
            None if result["best_pair"] is None else result["best_pair"]["sensitivity"]
        ),
        "pair_best_specificity": (
            None if result["best_pair"] is None else result["best_pair"]["specificity"]
        ),
        "record_best_threshold": (
            None if result["best_record"] is None else result["best_record"]["threshold"]
        ),
        "record_best_sensitivity": (
            None if result["best_record"] is None else result["best_record"]["sensitivity"]
        ),
        "record_best_specificity": (
            None if result["best_record"] is None else result["best_record"]["specificity"]
        ),
    }
    for dataset_name, result in dataset_runs.items()
)

summary_df

Scoring candidate pairs:   0%|          | 0/871678 [00:00<?, ?it/s]

2026-07-30 19:22:10.134 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.184 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.189 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.195 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.240 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.312 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.361 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.703 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.774 | WARNING  | app.dedupe:compare_authors:629 - One or both records have no authors.
2026-07-30 19:22:11.777 | WARNING  | 

Scoring candidate pairs:   0%|          | 0/8487 [00:00<?, ?it/s]

,dataset,n_records,n_pairs,pair_best_threshold,pair_best_sensitivity,pair_best_specificity,record_best_threshold,record_best_sensitivity,record_best_specificity
0,depression,79880,871678,0.8,0.993909,0.999925,0.80,0.990133,0.999326
1,diabetes,1845,8487,0.7,0.990215,0.994943,0.85,0.992070,0.998288


In [3]:
MIN_SENSITIVITY = 0.98
THRESHOLD_OVERRIDE = 0.85

if "dataset_runs" not in globals():
    raise RuntimeError("Run the dataset pipeline cell first.")

final_record_level_rows = []
for dataset_name, result in dataset_runs.items():
    record_metrics_df = result["record_metrics_df"]

    if THRESHOLD_OVERRIDE is not None:
        matched = record_metrics_df[
            record_metrics_df["threshold"].sub(THRESHOLD_OVERRIDE).abs() < 1e-12
        ]
        if matched.empty:
            raise ValueError(
                f"Threshold {THRESHOLD_OVERRIDE} not found for {dataset_name}. "
                f"Available thresholds: {sorted(record_metrics_df['threshold'].tolist())}"
            )
        chosen = matched.iloc[0]
        selection_note = "manual_threshold_override"
    else:
        chosen = find_best_threshold(record_metrics_df, MIN_SENSITIVITY)
        if chosen is None:
            chosen = record_metrics_df.loc[record_metrics_df["sensitivity"].idxmax()]
            selection_note = f"fallback_max_sensitivity_below_{MIN_SENSITIVITY:.2f}"
        else:
            selection_note = "max_specificity_with_sensitivity_floor"

    final_record_level_rows.append(
        {
            "dataset": dataset_name,
            "threshold": float(chosen["threshold"]),
            "TP": int(chosen["TP"]),
            "FP": int(chosen["FP"]),
            "TN": int(chosen["TN"]),
            "FN": int(chosen["FN"]),
            "sensitivity": float(chosen["sensitivity"]),
            "specificity": float(chosen["specificity"]),
            "precision": float(chosen["precision"]),
            "selection_rule": selection_note,
        }
    )

final_record_level_df = pd.DataFrame(final_record_level_rows).sort_values(
    ["threshold", "dataset"]
).reset_index(drop=True)
final_record_level_df

,dataset,threshold,TP,FP,TN,FN,sensitivity,specificity,precision,selection_rule
0,depression,0.85,9979,39,69706,156,0.984608,0.999441,0.996107,manual_threshold_override
1,diabetes,0.85,1251,1,583,10,0.992070,0.998288,0.999201,manual_threshold_override


In [ ]:
print("INTERCEPT:", SCORE_CONFIG.intercept)
print("SCORE_FIELDS:", SCORE_CONFIG.fields)
print("SCORE_WEIGHTS:", SCORE_CONFIG.weights)

INTERCEPT: -18.688678
SCORE_FIELDS: ['doi', 'title', 'authors', 'year', 'journal', 'pages', 'issue', 'intercept']
SCORE_WEIGHTS: {'doi': 5.304259, 'title': 11.591445, 'authors': 3.704393, 'year': 2.646572, 'journal': 3.179915, 'pages': 3.79124, 'issue': 1.153251, 'intercept': -18.688678}
